# IFRS 9 and CECL ECL

**Purpose:** Walk through a compact expected-credit-loss workflow using the `finstack_quant.statements_analytics` bindings.

**Prerequisites:** Familiarity with PD, LGD, EAD, and the difference between 12-month and lifetime ECL.

**In this notebook:** We classify a loan across Stage 1, Stage 2, and Stage 3 conditions, compute 12-month and lifetime ECL, and then add probability-weighted macro scenarios.


## Concept

Under IFRS 9, Stage 1 exposures use **12-month ECL**, while Stage 2 and Stage 3 exposures use **lifetime ECL**. In practice, the workflow is usually:

1. Define the exposure state.
2. Classify the stage from delinquency or credit deterioration.
3. Build a cumulative PD term structure.
4. Compute stage-appropriate ECL.
5. Weight multiple macro scenarios into a final allowance.


In [1]:
import sys
sys.path.insert(0, "../..")

from _shared import banner
from finstack_quant.statements_analytics import (
    Exposure,
    StagingConfig,
    classify_stage,
    compute_ecl,
    compute_ecl_weighted,
)

base_exposure = Exposure(
    id="CORP-LOAN-001",
    ead=1_000_000.0,
    lgd=0.45,
    eir=0.06,
    remaining_maturity=5.0,
    current_pd=0.032,
    origination_pd=0.030,
    dpd=0,
)

sicr_exposure = Exposure(
    id="CORP-LOAN-001-SICR",
    ead=1_000_000.0,
    lgd=0.45,
    eir=0.06,
    remaining_maturity=5.0,
    current_pd=0.045,
    origination_pd=0.030,
    dpd=0,
)

default_exposure = Exposure(
    id="CORP-LOAN-001-NPL",
    ead=1_000_000.0,
    lgd=0.45,
    eir=0.06,
    remaining_maturity=5.0,
    current_pd=0.25,
    origination_pd=0.030,
    dpd=120,
)

base_pd_schedule = [
    (0.0, 0.0),
    (1.0, 0.008),
    (2.0, 0.015),
    (3.0, 0.022),
    (4.0, 0.028),
    (5.0, 0.032),
]

sicr_pd_schedule = [
    (0.0, 0.0),
    (1.0, 0.015),
    (2.0, 0.025),
    (3.0, 0.033),
    (4.0, 0.040),
    (5.0, 0.045),
]

upside_pd_schedule = [
    (0.0, 0.0),
    (1.0, 0.004),
    (2.0, 0.008),
    (3.0, 0.012),
    (4.0, 0.016),
    (5.0, 0.020),
]

downside_pd_schedule = [
    (0.0, 0.0),
    (1.0, 0.025),
    (2.0, 0.045),
    (3.0, 0.065),
    (4.0, 0.085),
    (5.0, 0.100),
]

scenarios = [
    (0.60, base_pd_schedule),
    (0.20, upside_pd_schedule),
    (0.20, downside_pd_schedule),
]

base_exposure


,id,ead,lgd,eir,remaining_maturity,current_pd,origination_pd,dpd
0,CORP-LOAN-001,1000000.0,0.45,0.06,5.0,0.032,0.03,0


## Stage classification and allowance build

The next cell mirrors the flat example script, but in notebook form so you can inspect intermediate outputs. It shows the stage trigger for each exposure, compares Stage 1 versus Stage 2 allowance size, and then applies IFRS 9-style scenario weights.


In [2]:
banner("Synthetic exposure")
print(base_exposure)

banner("Stage classification")
for label, exposure in (
    ("Performing", base_exposure),
    ("SICR", sicr_exposure),
    ("90+ DPD", default_exposure),
):
    staging = classify_stage(exposure, StagingConfig(pd_delta_absolute=0.01))
    triggers = staging.triggers
    trigger_trail = ", ".join(triggers) if triggers else "none"
    print(f"{label:<12} -> {staging.stage.value:<8} | triggers: {trigger_trail}")

ecl_12m = compute_ecl(
    base_exposure,
    base_pd_schedule,
    stage="stage1",
    bucket_width_years=0.25,
).ecl

ecl_lifetime = compute_ecl(
    sicr_exposure,
    sicr_pd_schedule,
    stage="stage2",
    bucket_width_years=0.25,
).ecl

banner("ECL computation")
print(f"Stage 1 (12-month) ECL       : ${ecl_12m:,.2f}")
print(f"Stage 2 (lifetime) ECL       : ${ecl_lifetime:,.2f}")
print(f"Lifetime / 12m multiple      : {ecl_lifetime / ecl_12m:.1f}x")

weighted_12m = compute_ecl_weighted(base_exposure, scenarios, stage="stage1").ecl

weighted_lifetime = compute_ecl_weighted(base_exposure, scenarios, stage="stage2").ecl

banner("Probability-weighted macro scenarios")
print(f"Weighted Stage 1 ECL         : ${weighted_12m:,.2f}")
print(f"Weighted Stage 2 ECL         : ${weighted_lifetime:,.2f}")
print("\nPer-scenario Stage 1 ECL:")
for label, (weight, schedule) in zip(("base", "upside", "downside"), scenarios):
    scenario_ecl = compute_ecl(
        base_exposure,
        schedule,
        stage="stage1",
        bucket_width_years=0.25,
    ).ecl
    print(f"  {label:<8} (w={weight:.0%}) -> ${scenario_ecl:,.2f}")



Synthetic exposure
Exposure(id='CORP-LOAN-001', ead=1000000.00, lgd=0.4500, eir=0.0600, maturity=5.00y, current_pd=0.0320, origination_pd=0.0300, dpd=0)

Stage classification
Performing   -> Stage 1  | triggers: no_trigger
SICR         -> Stage 2  | triggers: pd_delta_absolute (delta=0.0150 > 0.0100)
90+ DPD      -> Stage 3  | triggers: dpd_stage3 (dpd=120 > 90)

ECL computation
Stage 1 (12-month) ECL       : $3,497.09
Stage 2 (lifetime) ECL       : $18,093.84
Lifetime / 12m multiple      : 5.2x

Probability-weighted macro scenarios
Weighted Stage 1 ECL         : $4,633.65
Weighted Stage 2 ECL         : $17,076.44

Per-scenario Stage 1 ECL:
  base     (w=60%) -> $3,497.09
  upside   (w=20%) -> $1,748.55
  downside (w=20%) -> $10,928.42


## Takeaways

- `classify_stage()` returns a `StageResult` with the full ordered trigger list, giving an explicit audit trail for the staging decision.
- `compute_ecl()` separates **term structure inputs** from **stage selection**.
- `compute_ecl_weighted()` is the clean bridge from single-scenario expected loss to IFRS 9 probability weighting.
- The main modeling judgment still lives outside the API: how you build the PD scenarios and when you call SICR.


In [3]:
{
    "stage1_ecl": round(ecl_12m, 2),
    "stage2_ecl": round(ecl_lifetime, 2),
    "weighted_stage1_ecl": round(weighted_12m, 2),
    "weighted_stage2_ecl": round(weighted_lifetime, 2),
}


{'stage1_ecl': 3497.09,
 'stage2_ecl': 18093.84,
 'weighted_stage1_ecl': 4633.65,
 'weighted_stage2_ecl': 17076.44}